# MovieLens recommender experiments

Explore the ratings, compare the existing SVD and PMF models, and inspect their
recommendations for individual users.

Open this notebook from the repository root and select the project's Python
environment as the Jupyter kernel. Install **ipykernel** in that environment
if the kernel is unavailable. Put **ratings.dat**, **movies.dat**, and **users.dat**
in **data/**. [MovieLens 1M download](https://grouplens.org/datasets/movielens/1m/)

Run the cells in order. After editing a model or utility module, restart the
kernel and rerun its dependent cells. Report export is opt-in in the last cell.

In [ ]:
import sys
from importlib.util import find_spec

required_packages = ("matplotlib", "numba", "numpy", "pandas", "scipy", "sklearn")
missing_packages = [package for package in required_packages if find_spec(package) is None]
if missing_packages:
    raise RuntimeError(
        f"The active kernel ({sys.executable}) is missing {', '.join(missing_packages)}. "
        "Select Python (matrix-factorization) in JupyterLab and rerun this cell."
    )

%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from models.pmf_model import PMFModel
from models.svd_model import SVDModel
from utils.matrix_creation import normalize_matrix
from utils.recommendation import (
    compare_recommendations,
    generate_recommendations,
    get_movie_data,
)

## Experiment settings

The seed and test fraction reproduce the shared 80/20 split in the model
scripts. Use the validation curve to compare PMF settings, then use the test
results for the final comparison.

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.20
VALIDATION_FRACTION = 0.10

SVD_FACTORS = 50
PMF_SETTINGS = {"k": 50, "learning_rate": 0.01, "reg": 0.05, "random_state": RANDOM_STATE}
PMF_MAX_EPOCHS = 45
PMF_PATIENCE = 10

## Load and inspect the data

The shared loader cleans the source files and retains the latest rating for
each user and movie pair. PMF validation uses 10% of the training ratings by
default.

In [ ]:
data_path = Path("data")
missing = [
    name for name in ("ratings.dat", "movies.dat", "users.dat")
    if not (data_path / name).is_file()
]
if missing:
    raise FileNotFoundError(f"Place these MovieLens 1M files in data/: {', '.join(missing)}")

ratings, movies = get_movie_data()
assert ratings["Rating"].between(1, 5).all()
assert not ratings.duplicated(subset=["UserID", "MovieID"]).any()

train, test = train_test_split(ratings, test_size=TEST_FRACTION, random_state=RANDOM_STATE)
pmf_train, validation = train_test_split(
    train, test_size=VALIDATION_FRACTION, random_state=RANDOM_STATE
)

splits = {"All": ratings, "Training": train, "PMF epoch selection": pmf_train,
          "PMF validation": validation, "Test": test}
split_summary = pd.DataFrame.from_dict(
    {
        name: {"Ratings": len(rows), "Users": rows["UserID"].nunique(),
               "Movies": rows["MovieID"].nunique()}
        for name, rows in splits.items()
    },
    orient="index",
)
display(split_summary)
display(ratings["Rating"].value_counts().sort_index().rename("Rating count").to_frame())

## Train SVD

Build the interaction matrix from the training ratings and use the existing
user-mean normalization and SVD model.

In [ ]:
matrix, user_means = normalize_matrix(
    train.pivot(index="UserID", columns="MovieID", values="Rating")
)
print(f"Training matrix: {matrix.shape[0]:,} users by {matrix.shape[1]:,} movies")
print(f"Observed ratings: {len(train) / matrix.size:.1%}")
svd = SVDModel(k=SVD_FACTORS).fit(matrix, user_means, train["Rating"].mean())

## Tune PMF and inspect convergence

Change the number of factors, learning rate, or regularization in the settings
cell. Rerun that cell and the cells from here onward to see the effect.
Rising validation MSE alongside falling training MSE indicates overfitting.

In [ ]:
pmf_validation = PMFModel(epochs=PMF_MAX_EPOCHS, **PMF_SETTINGS).fit(
    pmf_train, validation, patience=PMF_PATIENCE
)
best_epoch = pmf_validation.best_epoch
history = pd.DataFrame(
    {
        "Training MSE": pmf_validation.history,
        "Validation MSE": np.square(pmf_validation.validation_rmse_history),
    },
    index=pd.Index(range(1, len(pmf_validation.history) + 1), name="Epoch"),
)
display(history.round(5))
print(f"Selected epoch: {best_epoch} | Validation MSE: {history.loc[best_epoch, 'Validation MSE']:.5f}")

pmf_figure, ax = plt.subplots(figsize=(8, 4), layout="constrained")
ax.plot(history.index, history["Training MSE"], label="Training")
ax.plot(history.index, history["Validation MSE"], label="Validation")
ax.axvline(best_epoch, color="gray", linestyle="--", label="Selected epoch")
ax.set(xlabel="Epoch", ylabel="MSE", title="PMF convergence")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

## Refit PMF and compare test performance

Refit PMF for the selected number of epochs on the full training set. Both
models are evaluated on the same held-out ratings.

In [ ]:
pmf = PMFModel(epochs=best_epoch, **PMF_SETTINGS).fit(train)
svd_metrics = svd.evaluate(test)
pmf_metrics = pmf.evaluate(test)

scores = pd.DataFrame(
    {
        "SVD": {"MSE": svd_metrics["SVD_MSE"], "RMSE": svd_metrics["SVD_RMSE"]},
        "PMF": {"MSE": pmf_metrics["PMF_MSE"], "RMSE": pmf_metrics["PMF_RMSE"]},
    }
).T
display(scores.round(4))
improvement = 1 - scores.loc["PMF", "RMSE"] / scores.loc["SVD", "RMSE"]
print(f"PMF relative RMSE improvement over SVD: {improvement:.2%}")

rmse_figure, ax = plt.subplots(figsize=(5, 3.5), layout="constrained")
bars = ax.bar(scores.index, scores["RMSE"], color=["tab:blue", "tab:orange"])
ax.bar_label(bars, labels=[f"{score:.4f}" for score in scores["RMSE"]], padding=3)
ax.set(ylabel="Test RMSE", ylim=(0, scores["RMSE"].max() * 1.18))
plt.show()

In [ ]:
sample = test.sample(n=min(2_000, len(test)), random_state=RANDOM_STATE)
sample_svd = [svd.predict(user, movie) for user, movie in zip(sample["UserID"], sample["MovieID"])]
sample_pmf = pmf.predict_many(sample["UserID"], sample["MovieID"])

prediction_figure, axes = plt.subplots(
    1, 2, figsize=(9, 4), sharex=True, sharey=True, layout="constrained"
)
for ax, name, predictions in zip(axes, ("SVD", "PMF"), (sample_svd, sample_pmf)):
    ax.scatter(sample["Rating"], predictions, s=9, alpha=0.15)
    ax.plot([1, 5], [1, 5], color="gray", linestyle="--", linewidth=1)
    ax.set(title=name, xlabel="Actual rating", xticks=range(1, 6), yticks=range(1, 6))
axes[0].set_ylabel("Predicted rating")
prediction_figure.suptitle(f"Predicted vs. actual ratings on {len(sample):,} test rows")
plt.show()

## Inspect recommendations for a user

Change **USER_ID** and rerun the next two cells to inspect a different user.
The PMF raw score helps explain the order when displayed ratings tie at 5.

In [ ]:
USER_ID = 1
TOP_N = 10

if USER_ID not in svd.user_ids or USER_ID not in pmf.user_ids:
    raise ValueError(f"Choose a user present in both trained models. Example: {svd.user_ids[0]}")

favorites = (
    ratings.loc[ratings["UserID"] == USER_ID]
    .merge(movies[["MovieID", "Title", "Genres"]], on="MovieID", how="left")
    .sort_values(["Rating", "Timestamp"], ascending=False)
)
print(f"Highest-rated movies for user {USER_ID}")
display(favorites[["Title", "Genres", "Rating"]].head(TOP_N))

svd_recs = generate_recommendations(USER_ID, svd, TOP_N)
pmf_recs = generate_recommendations(USER_ID, pmf, TOP_N)
recs = pd.concat([svd_recs, pmf_recs], ignore_index=True)
recs.insert(1, "Rank", recs.groupby("Model").cumcount() + 1)
is_pmf = recs["Model"].eq("PMF")
recs.loc[is_pmf, "Raw PMF score"] = pmf.predict_many(
    recs.loc[is_pmf, "UserID"], recs.loc[is_pmf, "MovieID"], clip=False
).round(4)
display(recs[["Model", "Rank", "Title", "Genres", "PredictedRating", "Raw PMF score"]])
shared = set(svd_recs["MovieID"]) & set(pmf_recs["MovieID"])
print(f"Movies recommended by both models: {len(shared)}")

In [ ]:
user_test = test.loc[test["UserID"] == USER_ID]
if user_test.empty:
    print(f"User {USER_ID} has no held-out ratings in this split.")
else:
    user_sample = (
        user_test.sample(n=min(TOP_N, len(user_test)), random_state=RANDOM_STATE)
        .merge(movies[["MovieID", "Title"]], on="MovieID", how="left")
    )
    user_sample["SVD"] = [svd.predict(USER_ID, movie) for movie in user_sample["MovieID"]]
    user_sample["PMF"] = pmf.predict_many(user_sample["UserID"], user_sample["MovieID"])
    user_ax = (
        user_sample.set_index("Title")[["Rating", "SVD", "PMF"]]
        .rename(columns={"Rating": "Actual"})
        .plot.barh(figsize=(9, max(3, len(user_sample) * 0.48)))
    )
    user_ax.invert_yaxis()
    user_ax.set(title=f"Held-out ratings for user {USER_ID}", xlabel="Rating", ylabel="", xlim=(0, 5.2))
    user_ax.figure.tight_layout()
    plt.show()

## Export project artifacts

Set **EXPORT_REPORTS** to **True** to write the current results to **processed/**
and **reports/**. This replaces matching report files from previous runs.

In [ ]:
EXPORT_REPORTS = False

if EXPORT_REPORTS:
    processed_dir = Path("processed")
    reports_dir = Path("reports")
    processed_dir.mkdir(exist_ok=True)
    reports_dir.mkdir(exist_ok=True)

    matrix.round(3).to_csv(processed_dir / "user_item_matrix.csv")
    svd.save()
    pmf.save({**svd_metrics, **pmf_metrics}, validation_run=pmf_validation)
    rmse_figure.savefig(reports_dir / "rmse_comparison.png", dpi=150)
    prediction_figure.savefig(reports_dir / "predicted_vs_actual.png", dpi=150)
    compare_recommendations(USER_ID, svd, pmf, TOP_N)